In [1]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0))
import torch; print(torch.__version__); print(torch.cuda.is_available())

Device: cuda
GPU: NVIDIA GeForce RTX 5080
2.14.0+cu130
True


In [3]:
"""
00_build_manifest_and_folds.py   (Phases 0-2 of the v3 plan)

Creates, ONCE:
  * manifest_v3.csv   one row per scan (subject, session, run, age, sex, CDR, CDRSUM, MMSE, path)
  * folds_v3.csv      frozen subject -> fold map (stratified by age band, sex, ever-impaired)
  * bins_v3.json      age-bin scheme shared by every model (so heads have identical shape)
  * cohort_flow_v3.json  counts for Figure 1
  * PREREGISTRATION_v3.md  dated template (fill in BEFORE running any training)
  * environment_v3.json    software versions

Usage:  python 00_build_manifest_and_folds.py            (refuses to overwrite a frozen split)
        python 00_build_manifest_and_folds.py --force    (only if you really mean it)
"""
import argparse, glob, json, os, re, sys, zipfile, datetime, platform
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold
import config_v3 as C


def parse_name(path):
    m = re.search(r"sub-(OAS\d+)_sess?-d(\d+)(?:_run-(\d+))?", os.path.basename(path))
    if not m:
        raise ValueError(path)
    sid, day, run = m.group(1), m.group(2), m.group(3) or "01"
    return dict(subject_id=sid, day=int(day), run=run, mri_label=f"{sid}_MR_d{day}", file_path=path)


def build_manifest():
    files = sorted(glob.glob(os.path.join(str(C.PREPROC_DIR), "**", "*_preproc.npz"), recursive=True))
    if not files:
        sys.exit(f"No *_preproc.npz under {C.PREPROC_DIR}")
    ok = [f for f in files if zipfile.is_zipfile(f)]
    print(f"npz found {len(files)} | valid {len(ok)} | corrupt {len(files) - len(ok)}")
    rows, bad = [], 0
    for f in ok:
        try:
            rows.append(parse_name(f))
        except ValueError:
            bad += 1
    print("unparseable filenames:", bad)
    man = pd.DataFrame(rows)

    meta = (pd.read_csv(C.SCAN_META_CSV)[["Label", "Age", "M/F"]]
            .rename(columns={"Label": "mri_label", "Age": "age_meta", "M/F": "sex"})
            .drop_duplicates("mri_label"))
    cdr = (pd.read_csv(C.CDR_CSV)[["MRI Label", "age at visit", "CDRTOT", "CDRSUM", "MMSE"]]
           .rename(columns={"MRI Label": "mri_label", "age at visit": "age_cdr"})
           .drop_duplicates("mri_label"))
    man = man.merge(meta, on="mri_label", how="left").merge(cdr, on="mri_label", how="left")
    man["age"] = pd.to_numeric(man["age_cdr"], errors="coerce").fillna(pd.to_numeric(man["age_meta"], errors="coerce"))
    man = man.drop(columns=["age_meta", "age_cdr"])
    return man


def main(force):
    os.makedirs(C.OUT, exist_ok=True)
    if C.FOLDS_CSV.exists() and not force:
        sys.exit(f"{C.FOLDS_CSV} already exists and is FROZEN. Use --force only if you accept invalidating all results.")

    man = build_manifest()
    flow = {"scans_with_npz": int(len(man)), "subjects_with_npz": int(man.subject_id.nunique())}

    # --- Step 1.3: CDR alignment audit (rule used = exact MRI-label match in your precomputed CDR file)
    flow["scans_missing_age"] = int(man.age.isna().sum())
    flow["scans_missing_CDR"] = int(man.CDRTOT.isna().sum())
    flow["sessions_missing_CDR"] = int(man[man.CDRTOT.isna()].mri_label.nunique())
    man = man.dropna(subset=["age"]).reset_index(drop=True)
    # scans with no CDR cannot be labelled CN/impaired -> excluded from training AND longitudinal windows
    man["has_cdr"] = man.CDRTOT.notna()

    # --- Step 1.2: repeat audit (same session, >=2 runs). Runs are distinguished only by run number here.
    runs = man.groupby("mri_label").run.nunique()
    flow["sessions"] = int(len(runs))
    flow["sessions_with_repeat"] = int((runs >= 2).sum())
    flow["sessions_with_3plus_runs"] = int((runs >= 3).sum())
    man["n_runs"] = man.mri_label.map(man.groupby("mri_label").size())

    sess_per_subj = man.groupby("subject_id").mri_label.nunique()
    flow["subjects_3plus_sessions"] = int((sess_per_subj >= 3).sum())
    cn_scans = man[(man.CDRTOT == 0)]
    flow["CN_scans"] = int(len(cn_scans))
    flow["CN_subjects"] = int(cn_scans.subject_id.nunique())
    flow["impaired_scans"] = int((man.CDRTOT > 0).sum())

    # --- Step 2: subject-level stratified folds
    subj = man.sort_values("day").groupby("subject_id").agg(
        age0=("age", "first"), sex=("sex", "first"),
        ever_impaired=("CDRTOT", lambda s: int((s > 0).any())), n_scans=("age", "size")).reset_index()
    subj["band"] = pd.cut(subj.age0, [0, 65, 75, 200], labels=["<65", "65-75", ">75"]).astype(str)
    subj["strat"] = subj.band + "|" + subj.sex.fillna("NA").astype(str) + "|" + subj.ever_impaired.astype(str)
    vc = subj.strat.value_counts()
    small = vc[vc < C.N_FOLDS].index
    subj.loc[subj.strat.isin(small), "strat"] = "other"
    if (subj.strat == "other").sum() and (subj.strat == "other").sum() < C.N_FOLDS:
        subj.loc[subj.strat == "other", "strat"] = vc.index[0]
    skf = StratifiedKFold(C.N_FOLDS, shuffle=True, random_state=C.SEED)
    subj["fold"] = -1
    for k, (_, te) in enumerate(skf.split(subj, subj.strat)):
        subj.loc[subj.index[te], "fold"] = k
    assert (subj.fold >= 0).all()
    subj[["subject_id", "fold", "strat"]].to_csv(C.FOLDS_CSV, index=False)
    man = man.merge(subj[["subject_id", "fold"]], on="subject_id")
    # integrity: no subject in two folds
    assert man.groupby("subject_id").fold.nunique().max() == 1
    man.sort_values(["subject_id", "day", "run"]).to_csv(C.MANIFEST_CSV, index=False)

    # --- shared age-bin scheme from CN ages only
    lo = float(np.floor(cn_scans.age.min())) - 4
    hi = float(np.ceil(cn_scans.age.max())) + 4
    json.dump(dict(bin_min=lo, bin_max=hi, bin_step=C.BIN_STEP, sigma=C.SIGMA, n_bins=int((hi - lo) / C.BIN_STEP)),
              open(C.BINS_JSON, "w"), indent=2)

    flow["fold_subject_counts"] = subj.fold.value_counts().sort_index().to_dict()
    flow["fold_CN_scan_counts"] = man[man.CDRTOT == 0].fold.value_counts().sort_index().to_dict()
    flow["fold_impaired_subject_counts"] = subj[subj.ever_impaired == 1].fold.value_counts().sort_index().to_dict()
    json.dump(flow, open(C.OUT / "cohort_flow_v3.json", "w"), indent=2, default=int)
    print(json.dumps(flow, indent=2, default=int))

    # --- software versions (Step 3.4 / Phase 11 item 8)
    env = dict(python=platform.python_version(), platform=platform.platform(), seed=C.SEED,
               date=str(datetime.datetime.now()))
    for pkg in ("numpy", "pandas", "scipy", "sklearn", "torch", "statsmodels", "lifelines"):
        try:
            env[pkg] = __import__(pkg).__version__
        except Exception:
            env[pkg] = "not installed"
    json.dump(env, open(C.OUT / "environment_v3.json", "w"), indent=2)

    # --- pre-registration template (Step 0.3)
    pr = C.OUT / "PREREGISTRATION_v3.md"
    if not pr.exists():
        pr.write_text(f"""# Pre-registration (v3) -- written {datetime.date.today()} BEFORE training v3 models

NOTE: v2 test results were already seen. v3 design choices were therefore made on cross-validation /
validation data only; state this honestly in the paper.

1. Primary accuracy endpoint: out-of-fold MAE on CN scans (cross-fitted bias correction), with subject-cluster bootstrap CI.
2. Primary reliability endpoint: pooled within-session sigma_SEM and MDC, subject-cluster bootstrap CI (1000 resamples).
3. Fixed model set (do not change after seeing survival results): m0 (v2-style), m1 (v3 primary), m2 (m1 + consistency loss). Ensemble = seeds within fold.
4. Baseline window: first k=3 sessions, all CDR 0, >=1 later session. Minimum window span: ____ years (primary) / sensitivity ____.
5. Primary clinical endpoint rule: >=30 conversion events -> survival primary; otherwise continuous (CDR-SB/MMSE slope) primary.
6. Primary comparison: paired out-of-bag C-index difference, shrunk-slope vs raw-slope Cox model (95% bootstrap CI).
7. Covariates in floor model: baseline age, sex, baseline BAG.
8. Exclusion rules (image QC only, never error-based): ____
9. Known limitations to state: few converters, interval censoring, single cohort, affine-only registration.
""")
    print("\nFrozen folds saved to", C.FOLDS_CSV, "-- never regenerate.")


if __name__ == "__main__":
    # Bypass argparse in Jupyter by calling main() directly:
    main(force=False)


npz found 3337 | valid 3337 | corrupt 0
unparseable filenames: 0
{
  "scans_with_npz": 3337,
  "subjects_with_npz": 1098,
  "scans_missing_age": 1,
  "scans_missing_CDR": 319,
  "sessions_missing_CDR": 208,
  "sessions": 2113,
  "sessions_with_repeat": 976,
  "sessions_with_3plus_runs": 209,
  "subjects_3plus_sessions": 270,
  "CN_scans": 2321,
  "CN_subjects": 788,
  "impaired_scans": 697,
  "fold_subject_counts": {
    "0": 220,
    "1": 220,
    "2": 220,
    "3": 219,
    "4": 219
  },
  "fold_CN_scan_counts": {
    "0": 479,
    "1": 473,
    "2": 442,
    "3": 475,
    "4": 452
  },
  "fold_impaired_subject_counts": {
    "0": 68,
    "1": 68,
    "2": 68,
    "3": 67,
    "4": 67
  }
}

Frozen folds saved to v3/folds_v3.csv -- never regenerate.


In [6]:
"""
01_qc_scans.py   (Steps 1.4, 3.2 -- image-only QC; CPU, no GPU needed)

Why: in v2, 39 of 358 test CN scans (11%) had |BAG| > 10 y and a handful of same-day repeats differed by 7-13 y.
Errors that large are almost never biology; they are failed skull-strips / registrations. This script flags them
using IMAGE properties only (never the model's error -- error-based exclusion would be circular).

Per scan:   brain_frac (non-zero voxel fraction), z_mean, z_std, nan_frac, ncc_template (similarity to a CN mean template)
Per session with >=2 runs:   ncc_repeat (similarity between run 1 and run 2 after registration)
Flags:      qc_flag = 1 when a robust z-score (median/MAD) of ncc_template or brain_frac is beyond +-QC_Z, or ncc_repeat is low.

Output: v3/qc_scans_v3.csv  (merge on file_path).  ALWAYS eyeball ~30 flagged and ~30 unflagged scans before fixing exclusion rules.
"""
import argparse
import numpy as np, pandas as pd
from tqdm import tqdm
import config_v3 as C


def load(p):
    with np.load(p) as z:
        return z["data"].astype(np.float32)


def ncc(a, b):
    m = (a != 0) | (b != 0)
    x, y = a[m], b[m]
    x, y = x - x.mean(), y - y.mean()
    return float((x * y).sum() / (np.sqrt((x * x).sum() * (y * y).sum()) + 1e-8))


def robust_z(s):
    med = s.median()
    mad = (s - med).abs().median() * 1.4826 + 1e-9
    return (s - med) / mad


def main(qc_z, n_template):
    man = pd.read_csv(C.MANIFEST_CSV)
    rng = np.random.default_rng(C.SEED)
    cn = man[man.CDRTOT == 0]
    tmpl_paths = rng.choice(cn.file_path.values, size=min(n_template, len(cn)), replace=False)
    template = np.mean([load(p) for p in tqdm(tmpl_paths, desc="template")], axis=0)

    rows = []
    for p in tqdm(man.file_path, desc="per-scan QC"):
        v = load(p)
        rows.append(dict(file_path=p, brain_frac=float((v != 0).mean()), z_mean=float(v[v != 0].mean()) if (v != 0).any() else np.nan,
                         z_std=float(v[v != 0].std()) if (v != 0).any() else np.nan,
                         nan_frac=float(np.isnan(v).mean()), ncc_template=ncc(v, template)))
    qc = pd.DataFrame(rows)

    # same-session repeat NCC (run1 vs run2)
    rep = {}
    for lab, g in tqdm(man.sort_values("run").groupby("mri_label"), desc="repeat NCC"):
        if len(g) >= 2:
            rep[lab] = ncc(load(g.file_path.iloc[0]), load(g.file_path.iloc[1]))
    qc = qc.merge(man[["file_path", "mri_label"]], on="file_path")
    qc["ncc_repeat"] = qc.mri_label.map(rep)

    qc["z_ncc_template"] = robust_z(qc.ncc_template)
    qc["z_brain_frac"] = robust_z(qc.brain_frac)
    low_rep = qc.ncc_repeat < (qc.ncc_repeat.median() - qc_z * (qc.ncc_repeat - qc.ncc_repeat.median()).abs().median() * 1.4826)
    qc["qc_flag"] = ((qc.z_ncc_template < -qc_z) | (qc.z_brain_frac.abs() > qc_z) | (qc.nan_frac > 0) | low_rep.fillna(False)).astype(int)
    qc.to_csv(C.QC_CSV, index=False)

    print(f"scans {len(qc)} | flagged {qc.qc_flag.sum()} ({100 * qc.qc_flag.mean():.1f}%)")
    print(qc[["brain_frac", "ncc_template", "ncc_repeat"]].describe().round(3))
    print("Next: open the 30 lowest-ncc_template scans in a viewer; confirm they are failed registrations/skull-strips.")
    print("Then freeze the rule (QC_Z) in the pre-registration and apply it once.")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--qc_z", type=float, default=4.0)
    ap.add_argument("--n_template", type=int, default=100)
    a = ap.parse_args(args=[])
    main(a.qc_z, a.n_template)


repeat NCC: 100%|██████████| 2113/2113 [02:34<00:00, 13.64it/s]


scans 3336 | flagged 631 (18.9%)
       brain_frac  ncc_template  ncc_repeat
count    3336.000      3336.000    2199.000
mean        0.407         0.843       0.830
std         0.025         0.130       0.259
min         0.200         0.390       0.005
25%         0.407         0.874       0.735
50%         0.412         0.902       0.973
75%         0.417         0.914       0.981
max         0.523         0.934       0.991
Next: open the 30 lowest-ncc_template scans in a viewer; confirm they are failed registrations/skull-strips.
Then freeze the rule (QC_Z) in the pre-registration and apply it once.


In [11]:
"""
02_zero_shot_probe.py   (Step 3.5 -- ~15-30 min on the GPU, NO training)

Question: does the UK-Biobank SFCN "see" your preprocessed volumes in the intensity convention it was trained on?
The UKB model expects x / mean(x) (brain-extracted, background 0, mean over the whole volume).
Your v2 tensors are brain-only z-scores (negative inside dark tissue, mean 0). With blocks 1-4 frozen in v2, the network
could not adapt to that mismatch. This probe runs the *pretrained, untouched* network on CN scans of fold 0 under
several conventions and reports how well it ranks and predicts age. No labels are used for training.

Pick the convention with the highest Pearson r / lowest corrected MAE, then set MEANNORM_CV in config_v3.py.
(Fold 0 is consumed by this single low-capacity decision; state it in the methods.)
"""
import argparse, json
import numpy as np, pandas as pd, torch
from scipy.stats import pearsonr, linregress
import config_v3 as C
import core_v3 as K

MODES = [("zscore", 0.0), ("meannorm", 0.25), ("meannorm", 0.30), ("meannorm", 0.35), ("meannorm", 0.40), ("meannorm", 0.50)]


@torch.no_grad()
def run(model, store, df, mode, cv, centers, dev, bs=4):
    preds = []
    for i in range(0, len(df), bs):
        x = torch.stack([store.get(p) for p in df.file_path.iloc[i:i + bs]]).to(dev)
        x = K.to_input(x, mode, cv)
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=dev.type == "cuda"):
            lp = model(x)
        preds.extend(K.expected_age(lp, centers).cpu().numpy().tolist())
    return np.array(preds)


def main(n_scans):
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    man = pd.read_csv(C.MANIFEST_CSV)
    df = man[(man.fold == 0) & (man.CDRTOT == 0) & man.age.between(45, 80)]
    df = df.groupby("subject_id").head(1)                                   # one scan per subject
    df = df.sample(n=min(n_scans, len(df)), random_state=C.SEED).reset_index(drop=True)
    print("probe scans:", len(df), "| age range", df.age.min(), "-", df.age.max())

    store = K.ScanStore()
    store.add(df.file_path)
    model = K.load_pretrained_sfcn(C.SFCN_CKPT, K.OLD_N_BINS, K.OLD_MIN, K.OLD_STEP, device=dev).eval()
    centers = (K.OLD_MIN + K.OLD_STEP / 2 + torch.arange(K.OLD_N_BINS, dtype=torch.float32)).to(dev)

    rows = []
    for mode, cv in MODES:
        p = run(model, store, df, mode, cv, centers, dev)
        a = df.age.to_numpy()
        r = pearsonr(a, p)[0]
        s = linregress(a, p)
        corr = (p - s.intercept) / s.slope
        rows.append(dict(mode=mode, cv=cv, raw_MAE=np.abs(p - a).mean(), pearson_r=r, slope=s.slope,
                         MAE_after_linear_fit=np.abs(corr - a).mean()))
        print(rows[-1])
    out = pd.DataFrame(rows).sort_values("pearson_r", ascending=False)
    print("\n", out.round(3).to_string(index=False))
    out.to_csv(C.OUT / "zero_shot_probe.csv", index=False)
    best = out.iloc[0]
    print(f"\nBest by Pearson r: mode={best['mode']} cv={best['cv']}.")
    print("If 'zscore' wins, set input_mode='zscore' for m1/m2 in config_v3.VARIANTS; "
          "otherwise set MEANNORM_CV to the winning cv.")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--n_scans", type=int, default=150)
    args = ap.parse_args(args=[]) # Fix: Pass empty args list
    main(args.n_scans)


probe scans: 143 | age range 45.24 - 79.64


Caching scans:   0%|          | 0/143 [00:00<?, ?it/s]

Caching scans: 100%|██████████| 143/143 [00:11<00:00, 12.93it/s]


{'mode': 'zscore', 'cv': 0.0, 'raw_MAE': np.float64(9.011732405015639), 'pearson_r': np.float64(-0.40738861722205644), 'slope': np.float64(-0.010138284832709213), 'MAE_after_linear_fit': np.float64(14.896526433409903)}
{'mode': 'meannorm', 'cv': 0.25, 'raw_MAE': np.float64(10.248860954871544), 'pearson_r': np.float64(0.08422327336245129), 'slope': np.float64(0.007567061683123128), 'MAE_after_linear_fit': np.float64(65.16936034746732)}
{'mode': 'meannorm', 'cv': 0.3, 'raw_MAE': np.float64(10.190359237777603), 'pearson_r': np.float64(0.058961374724108), 'slope': np.float64(0.00559924105506802), 'MAE_after_linear_fit': np.float64(95.07521026785471)}
{'mode': 'meannorm', 'cv': 0.35, 'raw_MAE': np.float64(10.110742797851563), 'pearson_r': np.float64(0.04714367841632846), 'slope': np.float64(0.0045422752644093504), 'MAE_after_linear_fit': np.float64(122.44098202952077)}
{'mode': 'meannorm', 'cv': 0.4, 'raw_MAE': np.float64(10.001243460061666), 'pearson_r': np.float64(0.09049284730087677), 's

In [14]:
"""
03_train_cv.py   (Phase 4 -- the GPU-heavy step; resumable)

For every (variant, fold, seed):
    train on CN scans of the other folds (minus a 12 % subject-level early-stopping split),
    early-stop on that held-out CN split, save the best checkpoint.
Fold k's models are later used ONLY to predict fold k's subjects (script 04) => every scan gets an out-of-fold prediction.

Variants (config_v3.VARIANTS):
    m0  v2-style baseline (last block + head, z-score input, KL loss)
    m1  v3 primary (all blocks, layer-wise LR, tail re-weighting, KL + 0.1*L1, GPU augmentation)
    m2  m1 + consistency loss on same-day repeat pairs from TRAINING subjects only

Examples
    python 03_train_cv.py --variants m1 --folds 0 --seeds 1 --epochs 2     # smoke test, ~minutes
    python 03_train_cv.py --variants m0,m1,m2 --seeds 2                    # full run (hours; restart-safe)

Memory: if CUDA OOM at --bs 4, use --bs 2 --bn_mode freeze.
"""
import argparse, json, os, time
import numpy as np, pandas as pd, torch
from torch.utils.data import DataLoader, WeightedRandomSampler
import config_v3 as C
import core_v3 as K


def split_train_val(df_train_cn, seed, frac):
    subj = np.array(sorted(df_train_cn.subject_id.unique()))
    rng = np.random.default_rng(seed)
    val_s = set(rng.choice(subj, size=max(5, int(len(subj) * frac)), replace=False))
    is_val = df_train_cn.subject_id.isin(val_s)
    return df_train_cn[~is_val].reset_index(drop=True), df_train_cn[is_val].reset_index(drop=True)


@torch.no_grad()
def evaluate(model, loader, cfg, centers, dev, cv):
    model.eval()
    err, n = 0.0, 0
    for x, age in loader:
        x, age = x.to(dev, non_blocking=True), age.to(dev)
        x = K.to_input(x, cfg["input_mode"], cv)
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=dev.type == "cuda"):
            lp = model(x)
        err += (K.expected_age(lp, centers) - age).abs().sum().item()
        n += len(age)
    return err / max(n, 1)


def train_one(variant, fold, seed, man, store, bins, args, dev):
    cfg = C.VARIANTS[variant]
    tag = f"{variant}_f{fold}_s{seed}"
    ckpt, meta = C.CKPT_DIR / f"{tag}.pt", C.CKPT_DIR / f"{tag}.json"
    if ckpt.exists() and meta.exists():
        print(f"[skip] {tag} already trained")
        return
    torch.manual_seed(seed * 1000 + fold)
    np.random.seed(seed * 1000 + fold)

    cn_train_pool = man[(man.fold != fold) & (man.CDRTOT == 0)]
    tr, va = split_train_val(cn_train_pool, C.SEED + fold, C.VAL_FRACTION)
    # Phase-11 integrity checks
    assert not (set(tr.subject_id) & set(va.subject_id))
    assert (tr.fold != fold).all() and (va.fold != fold).all(), "held-out fold leaked into training"
    store.add(pd.concat([tr, va]).file_path)

    centers = K.bin_centers(bins, dev)
    model = K.load_pretrained_sfcn(C.SFCN_CKPT, bins["n_bins"], bins["bin_min"], bins["bin_step"], device=dev)
    groups, trainable = K.configure_trainable(model, cfg["unfreeze"], cfg["lr_head"], cfg["lr_backbone"], cfg["llrd"])
    opt = torch.optim.AdamW(groups, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=3, min_lr=1e-7)

    train_ds, val_ds = K.ScanDS(store, tr), K.ScanDS(store, va)
    if cfg["reweight"]:
        sampler = WeightedRandomSampler(K.tail_weights(tr.age.values), num_samples=len(tr), replacement=True)
        tl = DataLoader(train_ds, batch_size=args.bs, sampler=sampler, num_workers=args.workers, pin_memory=True, drop_last=True)
    else:
        tl = DataLoader(train_ds, batch_size=args.bs, shuffle=True, num_workers=args.workers, pin_memory=True, drop_last=True)
    vl = DataLoader(val_ds, batch_size=args.bs, shuffle=False, num_workers=args.workers, pin_memory=True)

    pair_iter = None
    if cfg["consistency"] > 0:
        pds = K.PairDS(store, tr)                          # TRAINING subjects only
        print(f"  consistency pairs: {len(pds)}")
        pl = DataLoader(pds, batch_size=max(1, args.bs // 2), shuffle=True, num_workers=0, drop_last=True)
        pair_iter = iter(pl)

    best, bad, hist = 1e9, 0, []
    warm_steps = max(1, len(tl))                           # 1 epoch linear warm-up
    step = 0
    print(f"\n=== {tag} | train scans {len(tr)} | val scans {len(va)} | trainable: {trainable}")
    for ep in range(args.epochs):
        t0 = time.time()
        K.set_train_mode(model, trainable, args.bn_mode)
        tot, n = 0.0, 0
        for x, age in tl:
            if ep == 0:                                    # warm-up
                f = min(1.0, (step + 1) / warm_steps)
                for g in opt.param_groups:
                    g["lr"] = g["initial_lr"] * f
            x, age = x.to(dev, non_blocking=True), age.to(dev)
            x = K.gpu_augment(K.to_input(x, cfg["input_mode"], args.cv))
            y = K.soft_labels(age, bins, dev)
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=torch.bfloat16, enabled=dev.type == "cuda"):
                lp = model(x)
            loss = K.kl_loss(lp, y)
            if cfg["l1"] > 0:
                loss = loss + cfg["l1"] * (K.expected_age(lp, centers) - age).abs().mean()
            loss.backward()

            if pair_iter is not None and step % 2 == 0:    # consistency step on repeat pairs
                try:
                    xp, agep = next(pair_iter)
                except StopIteration:
                    pair_iter = iter(pl)
                    xp, agep = next(pair_iter)
                B = xp.shape[0]
                xp = xp.to(dev).view(B * 2, 1, *xp.shape[-3:])
                agep = agep.to(dev).repeat_interleave(2)
                xp = K.gpu_augment(K.to_input(xp, cfg["input_mode"], args.cv))
                with torch.autocast("cuda", dtype=torch.bfloat16, enabled=dev.type == "cuda"):
                    lpp = model(xp)
                ea = K.expected_age(lpp, centers).view(B, 2)
                closs = K.kl_loss(lpp, K.soft_labels(agep, bins, dev)) + cfg["consistency"] * ((ea[:, 0] - ea[:, 1]) ** 2).mean()
                closs.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
            tot += loss.item() * len(age)
            n += len(age)
            step += 1

        val_mae = evaluate(model, vl, cfg, centers, dev, args.cv)
        if ep >= 1:
            sched.step(val_mae)
        hist.append(dict(epoch=ep + 1, train_loss=tot / max(n, 1), val_mae=val_mae))
        print(f"ep {ep + 1:02d}/{args.epochs} | loss {tot / max(n, 1):.3f} | val MAE {val_mae:.3f} | "
              f"lr {opt.param_groups[0]['lr']:.2e} | {time.time() - t0:.0f}s")
        if val_mae < best:
            best, bad = val_mae, 0
            torch.save(model.state_dict(), ckpt)
        else:
            bad += 1
            if bad >= args.patience:
                print("early stop")
                break
        torch.cuda.empty_cache()

    json.dump(dict(tag=tag, best_val_mae=best, input_mode=cfg["input_mode"], cv=args.cv, history=hist,
                   n_train=len(tr), n_val=len(va), val_subjects=sorted(va.subject_id.unique())), open(meta, "w"), indent=1)
    print(f"{tag} done | best val MAE {best:.3f}")
    del model, opt
    torch.cuda.empty_cache()


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--variants", default="m0,m1,m2")
    ap.add_argument("--folds", default=",".join(map(str, range(C.N_FOLDS))))
    ap.add_argument("--seeds", type=int, default=2)
    ap.add_argument("--epochs", type=int, default=40)
    ap.add_argument("--patience", type=int, default=8)
    ap.add_argument("--bs", type=int, default=4)
    ap.add_argument("--workers", type=int, default=4)
    ap.add_argument("--bn_mode", default="train", choices=["train", "freeze"])
    ap.add_argument("--cv", type=float, default=C.MEANNORM_CV)
    args = ap.parse_args(args=[]) 

    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    os.makedirs(C.CKPT_DIR, exist_ok=True)
    man = pd.read_csv(C.MANIFEST_CSV)
    bins = K.load_bins(C.BINS_JSON)
    qc = pd.read_csv(C.QC_CSV) if C.QC_CSV.exists() else None
    if qc is not None:                                     # image-based exclusion applied to TRAINING scans only
        bad = set(qc[qc.qc_flag == 1].file_path)
        man["qc_flag"] = man.file_path.isin(bad).astype(int)
    store = K.ScanStore()
    for v in args.variants.split(","):
        for f in map(int, args.folds.split(",")):
            for s in range(args.seeds if v != "m0" else 1):
                m = man
                if qc is not None and "qc_flag" in man:
                    # exclude QC-failed scans from TRAINING/VALIDATION (they are still predicted & reported separately)
                    m = man[man.qc_flag == 0]
                train_one(v, f, s, m, store, bins, args, dev)


if __name__ == "__main__":
    main()


Caching scans: 100%|██████████| 1527/1527 [01:57<00:00, 13.01it/s]



=== m0_f0_s0 | train scans 1351 | val scans 176 | trainable: ['classifier.conv_6', 'feature_extractor.conv_5']
ep 01/40 | loss 3.281 | val MAE 7.783 | lr 1.00e-04 | 20s
ep 02/40 | loss 2.733 | val MAE 6.729 | lr 1.00e-04 | 17s
ep 03/40 | loss 2.615 | val MAE 6.435 | lr 1.00e-04 | 17s
ep 04/40 | loss 2.547 | val MAE 6.260 | lr 1.00e-04 | 17s
ep 05/40 | loss 2.488 | val MAE 6.175 | lr 1.00e-04 | 17s
ep 06/40 | loss 2.479 | val MAE 6.098 | lr 1.00e-04 | 17s
ep 07/40 | loss 2.415 | val MAE 6.004 | lr 1.00e-04 | 17s
ep 08/40 | loss 2.376 | val MAE 5.968 | lr 1.00e-04 | 17s
ep 09/40 | loss 2.339 | val MAE 5.940 | lr 1.00e-04 | 17s
ep 10/40 | loss 2.332 | val MAE 5.886 | lr 1.00e-04 | 17s
ep 11/40 | loss 2.304 | val MAE 5.851 | lr 1.00e-04 | 17s
ep 12/40 | loss 2.317 | val MAE 5.843 | lr 1.00e-04 | 17s
ep 13/40 | loss 2.287 | val MAE 5.841 | lr 1.00e-04 | 17s
ep 14/40 | loss 2.249 | val MAE 5.840 | lr 1.00e-04 | 17s
ep 15/40 | loss 2.241 | val MAE 5.738 | lr 1.00e-04 | 17s
ep 16/40 | loss 2.

Caching scans: 100%|██████████| 369/369 [00:27<00:00, 13.19it/s]


=== m0_f1_s0 | train scans 1317 | val scans 163 | trainable: ['classifier.conv_6', 'feature_extractor.conv_5']


ep 01/40 | loss 3.291 | val MAE 11.526 | lr 1.00e-04 | 17s
ep 02/40 | loss 2.764 | val MAE 10.170 | lr 1.00e-04 | 16s
ep 03/40 | loss 2.645 | val MAE 9.538 | lr 1.00e-04 | 16s
ep 04/40 | loss 2.546 | val MAE 9.573 | lr 1.00e-04 | 17s
ep 05/40 | loss 2.499 | val MAE 9.169 | lr 1.00e-04 | 16s
ep 06/40 | loss 2.473 | val MAE 9.022 | lr 1.00e-04 | 17s
ep 07/40 | loss 2.416 | val MAE 8.859 | lr 1.00e-04 | 16s
ep 08/40 | loss 2.387 | val MAE 8.735 | lr 1.00e-04 | 17s
ep 09/40 | loss 2.374 | val MAE 8.605 | lr 1.00e-04 | 17s
ep 10/40 | loss 2.354 | val MAE 8.419 | lr 1.00e-04 | 16s
ep 11/40 | loss 2.352 | val MAE 8.382 | lr 1.00e-04 | 16s
ep 12/40 | loss 2.338 | val MAE 8.392 | lr 1.00e-04 | 17s
ep 13/40 | loss 2.304 | val MAE 8.289 | lr 1.00e-04 | 17s
ep 14/40 | loss 2.266 | val MAE 8.193 | lr 1.00e-04 | 17s
ep 15/40 | loss 2.278 | val MAE 8.085 | lr 1.00e-04 | 17s
ep 16/40 | loss 2.292 | val MAE 8.035 | lr 1.00e-04 | 17s
ep 17/40 | loss 2.272 | val MAE 7.949 | lr 1.00e-04 | 16s
ep 18/40 | l

Caching scans: 0it [00:00, ?it/s]

... (64 outputs left out)

ep 01/40 | loss 3.898 | val MAE 5.625 | lr 3.00e-04 | 68s
ep 02/40 | loss 3.175 | val MAE 5.273 | lr 3.00e-04 | 68s
ep 03/40 | loss 2.814 | val MAE 4.479 | lr 3.00e-04 | 68s
ep 04/40 | loss 2.723 | val MAE 3.974 | lr 3.00e-04 | 68s
ep 05/40 | loss 2.560 | val MAE 4.107 | lr 3.00e-04 | 68s
ep 06/40 | loss 2.395 | val MAE 3.649 | lr 3.00e-04 | 68s
ep 07/40 | loss 2.182 | val MAE 3.805 | lr 3.00e-04 | 68s
ep 08/40 | loss 2.089 | val MAE 3.350 | lr 3.00e-04 | 68s
ep 09/40 | loss 1.992 | val MAE 3.639 | lr 3.00e-04 | 68s
ep 10/40 | loss 1.945 | val MAE 3.444 | lr 3.00e-04 | 68s
ep 11/40 | loss 1.985 | val MAE 3.325 | lr 3.00e-04 | 68s
ep 12/40 | loss 1.838 | val MAE 3.369 | lr 3.00e-04 | 69s
ep 13/40 | loss 1.874 | val MAE 3.211 | lr 3.00e-04 | 68s
ep 14/40 | loss 1.749 | val MAE 3.296 | lr 3.00e-04 | 68s
ep 15/40 | loss 1.669 | val MAE 3.132 | lr 3.00e-04 | 68s
ep 16/40 | loss 1.654 | val MAE 3.147 | lr 3.00e-04 | 68s
ep 17/40 | loss 1.689 | val MAE 3.375 | lr 3.00e-04 | 68s
ep 18/40 | los

Caching scans: 0it [00:00, ?it/s]

  consistency pairs: 377

=== m2_f4_s1 | train scans 1339 | val scans 185 | trainable: ['classifier.conv_6', 'feature_extractor.conv_5', 'feature_extractor.conv_4', 'feature_extractor.conv_3', 'feature_extractor.conv_2', 'feature_extractor.conv_1', 'feature_extractor.conv_0']


ep 01/40 | loss 3.885 | val MAE 5.715 | lr 3.00e-04 | 68s
ep 02/40 | loss 3.059 | val MAE 5.186 | lr 3.00e-04 | 68s
ep 03/40 | loss 2.840 | val MAE 4.247 | lr 3.00e-04 | 68s
ep 04/40 | loss 2.734 | val MAE 4.169 | lr 3.00e-04 | 68s
ep 05/40 | loss 2.518 | val MAE 3.866 | lr 3.00e-04 | 68s
ep 06/40 | loss 2.353 | val MAE 3.753 | lr 3.00e-04 | 68s
ep 07/40 | loss 2.234 | val MAE 3.657 | lr 3.00e-04 | 68s
ep 08/40 | loss 2.136 | val MAE 3.654 | lr 3.00e-04 | 68s
ep 09/40 | loss 2.036 | val MAE 3.548 | lr 3.00e-04 | 68s
ep 10/40 | loss 1.958 | val MAE 3.462 | lr 3.00e-04 | 68s
ep 11/40 | loss 1.915 | val MAE 3.267 | lr 3.00e-04 | 68s
ep 12/40 | loss 1.869 | val MAE 3.206 | lr 3.00e-04 | 68s
ep 13/40 | loss 1.789 | val MAE 3.335 | lr 3.00e-04 | 68s
ep 14/40 | loss 1.812 | val MAE 3.382 | lr 3.00e-04 | 68s
ep 15/40 | loss 1.769 | val MAE 3.341 | lr 3.00e-04 | 68s
ep 16/40 | loss 1.724 | val MAE 3.214 | lr 1.50e-04 | 68s
ep 17/40 | loss 1.611 | val MAE 3.309 | lr 1.50e-04 | 68s
ep 18/40 | los

In [17]:
"""
04a_predict_oof.py   (Phase 4.4-4.5 -- GPU)

Cross-fitted prediction: each scan is predicted ONLY by the models whose training set excluded that scan's subject
(fold k models -> fold k subjects). This applies equally to CN and impaired subjects, so noise comparisons between CDR
groups are not confounded by different ensemble sizes, and survival analyses can use the WHOLE cohort without leakage.

Per scan we store:
    predicted_age       mean over (seeds x TTA views)
    sd_seed             std across seed-means           (epistemic spread; NaN if only 1 seed)
    sd_tta              mean over seeds of std across views (augmentation spread)
    predicted_age_sd    std over all seed x view predictions (what 05_noise_floor uses for calibration)

Output: v3/predictions/raw_<variant>.csv
"""
import argparse, json, os
import numpy as np, pandas as pd, torch
import config_v3 as C
import core_v3 as K


@torch.no_grad()
def predict_models(models, df, store, cfg, cv, centers, dev, shifts, bs=4):
    """returns array (n_models, n_views, n_scans)"""
    out = np.zeros((len(models), len(shifts), len(df)), dtype=np.float32)
    for i in range(0, len(df), bs):
        x = torch.stack([store.get(p) for p in df.file_path.iloc[i:i + bs]]).to(dev)
        x = K.to_input(x, cfg["input_mode"], cv)
        for si, sh in enumerate(shifts):
            xs = K.shift_volume(x, sh)
            for mi, m in enumerate(models):
                with torch.autocast("cuda", dtype=torch.bfloat16, enabled=dev.type == "cuda"):
                    lp = m(xs)
                out[mi, si, i:i + len(xs)] = K.expected_age(lp, centers).cpu().numpy()
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--variants", default="m0,m1,m2")
    ap.add_argument("--n_tta", type=int, default=len(C.TTA_SHIFTS))
    ap.add_argument("--cv", type=float, default=C.MEANNORM_CV)
    args = ap.parse_args(args=[])

    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    os.makedirs(C.PRED_DIR, exist_ok=True)
    man = pd.read_csv(C.MANIFEST_CSV)
    bins = K.load_bins(C.BINS_JSON)
    centers = K.bin_centers(bins, dev)
    store = K.ScanStore()
    shifts = C.TTA_SHIFTS[:args.n_tta]

    for v in args.variants.split(","):
        cfg = C.VARIANTS[v]
        parts = []
        for k in range(C.N_FOLDS):
            ck = sorted(C.CKPT_DIR.glob(f"{v}_f{k}_s*.pt"))
            if not ck:
                print(f"[warn] no checkpoints for {v} fold {k}; skipping")
                continue
            models = []
            for c in ck:
                m = K.load_pretrained_sfcn(C.SFCN_CKPT, bins["n_bins"], bins["bin_min"], bins["bin_step"], device=dev)
                sd = torch.load(c, map_location=dev)
                m.load_state_dict({kk.replace("_orig_mod.", ""): vv for kk, vv in sd.items()})
                models.append(m.eval())
            df = man[man.fold == k].reset_index(drop=True)
            store.arr.clear()
            store.add(df.file_path, desc=f"{v} fold {k}")
            P = predict_models(models, df, store, cfg, args.cv, centers, dev, shifts)      # (M,V,N)
            seed_means = P.mean(1)                                                         # (M,N)
            df["predicted_age"] = P.mean((0, 1))
            df["sd_seed"] = seed_means.std(0) if len(models) > 1 else np.nan
            df["sd_tta"] = P.std(1).mean(0)
            df["predicted_age_sd"] = P.reshape(-1, P.shape[-1]).std(0)
            df["n_models"] = len(models)
            parts.append(df)
            print(f"{v} fold {k}: {len(df)} scans, {len(models)} model(s) x {len(shifts)} views")
        pd.concat(parts).to_csv(C.PRED_DIR / f"raw_{v}.csv", index=False)
        print("saved", C.PRED_DIR / f"raw_{v}.csv")


if __name__ == "__main__":
    main()


m0 fold 0: 100%|██████████| 680/680 [00:50<00:00, 13.40it/s]


m0 fold 0: 680 scans, 1 model(s) x 5 views


m0 fold 1: 100%|██████████| 645/645 [00:47<00:00, 13.51it/s]


m0 fold 1: 645 scans, 1 model(s) x 5 views


m0 fold 2: 100%|██████████| 658/658 [00:49<00:00, 13.40it/s]


m0 fold 2: 658 scans, 1 model(s) x 5 views


... (20 outputs left out)

m2 fold 3: 100%|██████████| 697/697 [00:51<00:00, 13.42it/s]


m2 fold 3: 697 scans, 2 model(s) x 5 views


m2 fold 4: 100%|██████████| 656/656 [00:48<00:00, 13.44it/s]


m2 fold 4: 656 scans, 2 model(s) x 5 views
saved v3/predictions/raw_m2.csv


In [29]:
"""
04b_bias_correct_accuracy.py   (Phase 5 + Step 4.7 -- CPU)

Input : v3/predictions/raw_<variant>.csv   (needs: subject_id, mri_label, run, age, CDRTOT, fold, predicted_age, predicted_age_sd)
Output: v3/predictions/bag_<variant>.csv    (adds brain_age, BAG)   +   v3/results/accuracy_<variant>.json / .csv

Bias correction (the v2 bug):  v2 fitted  pred = a + b*age  on TRAINING scans, i.e. on data the network had already
fit. In-sample the slope is ~1, on unseen subjects it was 0.56 -> the correction under-corrected and left a
BAG-vs-age trend of -0.24 years/year. Here the line is fitted on OUT-OF-FOLD CN predictions of the OTHER folds
("cross-fitted"), after averaging repeat scans within each session, and then applied unchanged to the held-out fold.

Usage: python 04b_bias_correct_accuracy.py --variant m1 [--raw path.csv] [--exclude_qc]
"""
import argparse, json
import numpy as np, pandas as pd
from scipy.stats import linregress, pearsonr
import config_v3 as C


def crossfit_correct(df, method="invert"):
    """method='invert'      : brain_age = (pred - a) / b   (v2 / Peng-style). Uses NO chronological age at test time,
                                so MAE / r / R2 computed from it are legitimate accuracy metrics.
       method='bag_regress' : BAG_corr = (pred - age) - (a' + b'*age)   (Beheshti-style residualisation).
                                Uses the subject's TRUE age to build the prediction => brain_age = age + residual.
                                Good for an age-independent BAG in downstream models, but its MAE is NOT an accuracy
                                metric (it is inflated by construction; see de Lange & Cole 2020, ref [12]). Never report it."""
    df = df.copy()
    df["brain_age"] = np.nan
    params = {}
    sess = (df[df.CDRTOT == 0].groupby(["fold", "mri_label"])
            .agg(age=("age", "mean"), pred=("predicted_age", "mean")).reset_index())
    for k in sorted(df.fold.unique()):
        tr = sess[sess.fold != k]
        m = df.fold == k
        if method == "invert":
            lr = linregress(tr.age, tr.pred)
            assert abs(lr.slope) > 1e-6
            df.loc[m, "brain_age"] = (df.loc[m, "predicted_age"] - lr.intercept) / lr.slope
        else:
            lr = linregress(tr.age, tr.pred - tr.age)
            df.loc[m, "brain_age"] = df.loc[m, "age"] + (df.loc[m, "predicted_age"] - df.loc[m, "age"]) - (lr.intercept + lr.slope * df.loc[m, "age"])
        params[int(k)] = dict(slope=lr.slope, intercept=lr.intercept, n_sessions=int(len(tr)))
    df["BAG"] = df.brain_age - df.age
    return df, params


def band_table(cn):
    rows = []
    for lo, hi in C.AGE_BANDS:
        g = cn[(cn.age >= lo) & (cn.age < hi)]
        if len(g):
            rows.append(dict(band=f"{lo}-{hi}", n=len(g), MAE=g.BAG.abs().mean(), mean_BAG=g.BAG.mean()))
    return pd.DataFrame(rows)


def three_class(df):
    t = np.digitize(df.age, C.CLASS_CUTS)
    p = np.digitize(df.brain_age, C.CLASS_CUTS)
    return float((t == p).mean()), float(pd.Series(t).value_counts(normalize=True).max())


def cluster_boot(df, fn, n=1000, seed=C.SEED):
    rng = np.random.default_rng(seed)
    groups = {s: g for s, g in df.groupby("subject_id")}
    keys = np.array(list(groups))
    vals = []
    for _ in range(n):
        samp = rng.choice(keys, len(keys), replace=True)
        vals.append(fn(pd.concat([groups[s] for s in samp])))
    return np.percentile(vals, [2.5, 97.5])


def main(variant, raw, exclude_qc, method):
    raw = raw or (C.PRED_DIR / f"raw_{variant}.csv")
    df = pd.read_csv(raw)
    if exclude_qc and C.QC_CSV.exists():
        qc = pd.read_csv(C.QC_CSV)[["file_path", "qc_flag"]]
        df = df.merge(qc, on="file_path", how="left")
        print("excluding", int((df.qc_flag == 1).sum()), "image-QC-flagged scans (image-based rule only)")
        df = df[df.qc_flag != 1]
    df_alt, _ = crossfit_correct(df, "bag_regress" if method == "invert" else "invert")
    alt_mae = float(df_alt[df_alt.CDRTOT == 0].BAG.abs().mean())
    df, params = crossfit_correct(df, method)
    out = C.PRED_DIR / f"bag_{variant}.csv"
    out.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out, index=False)

    cn = df[df.CDRTOT == 0]
    res = dict(variant=variant, n_CN_scans=len(cn), n_CN_subjects=int(cn.subject_id.nunique()))
    res["MAE"] = float(cn.BAG.abs().mean())
    res["MAE_CI95"] = cluster_boot(cn, lambda d: d.BAG.abs().mean()).tolist()
    res["correction_method"] = method
    res["MAE_alt_method_NOT_REPORTABLE"] = alt_mae
    res["RMSE"] = float(np.sqrt((cn.BAG ** 2).mean()))
    res["pearson_r"] = float(pearsonr(cn.age, cn.brain_age)[0])
    res["R2"] = float(1 - ((cn.age - cn.brain_age) ** 2).sum() / ((cn.age - cn.age.mean()) ** 2).sum())
    res["raw_MAE_uncorrected"] = float((cn.predicted_age - cn.age).abs().mean())
    res["median_abs_error"] = float(cn.BAG.abs().median())
    res["frac_abs_BAG_gt10"] = float((cn.BAG.abs() > 10).mean())
    s = linregress(cn.age, cn.BAG)
    res["BAG_vs_age_slope"], res["BAG_vs_age_p"] = float(s.slope), float(s.pvalue)
    res["mean_BAG_CN"] = float(cn.BAG.mean())
    acc, maj = three_class(cn)
    res["age_group_acc_3class"], res["age_group_majority_baseline"] = acc, maj
    res["age_group_acc_CI95"] = cluster_boot(cn, lambda d: three_class(d)[0]).tolist()
    sess = cn.groupby("mri_label").agg(age=("age", "first"), BAG=("BAG", "mean"), subject_id=("subject_id", "first"))
    res["session_averaged_MAE"] = float(sess.BAG.abs().mean())
    res["bias_params_per_fold"] = params
    bands = band_table(cn)
    imp = df[df.CDRTOT > 0].groupby("CDRTOT").BAG.agg(["count", "mean", "std"]).reset_index()
    cdr0 = df[df.CDRTOT == 0].BAG.agg(["count", "mean", "std"])

    print(json.dumps({k: v for k, v in res.items() if k != "bias_params_per_fold"}, indent=2))
    print("\nBy age band (CN):\n", bands.round(2).to_string(index=False))
    print("\nBAG by CDR (should increase with CDR):\n", imp.round(2).to_string(index=False), "\nCDR 0 mean BAG:", round(cdr0["mean"], 2))
    if method == "bag_regress":
        print("\n[NOTE] --method bag_regress: MAE/R2 above are NOT valid accuracy metrics (true age enters the prediction). "
              "Report accuracy from --method invert; use bag_regress BAG only as an age-independent covariate.")
    if res["BAG_vs_age_p"] < 0.05 and method == "invert":
        print("\n[CHECK FAILED] BAG still trends with age in CN (Step 5.3). Fix before survival analyses.")
    if exclude_qc is False and res["frac_abs_BAG_gt10"] > 0.03:
        print(f"[WARN] {100 * res['frac_abs_BAG_gt10']:.1f}% of CN scans have |BAG|>10 y -> inspect preprocessing/QC.")

    C.RES_DIR.mkdir(parents=True, exist_ok=True)
    json.dump(res, open(C.RES_DIR / f"accuracy_{variant}.json", "w"), indent=2)
    bands.to_csv(C.RES_DIR / f"accuracy_bands_{variant}.csv", index=False)


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--variant", default="m1")
    ap.add_argument("--raw", default=None)
    ap.add_argument("--exclude_qc", action="store_true")
    ap.add_argument("--method", default="invert", choices=["invert", "bag_regress"])
    a = ap.parse_args(args=["--exclude_qc"])
    main(a.variant, a.raw, a.exclude_qc, a.method)


excluding 631 image-QC-flagged scans (image-based rule only)
{
  "variant": "m1",
  "n_CN_scans": 1896,
  "n_CN_subjects": 767,
  "MAE": 3.7646565741252074,
  "MAE_CI95": [
    3.5570406862318085,
    3.9793526903278527
  ],
  "correction_method": "invert",
  "MAE_alt_method_NOT_REPORTABLE": 2.814516856543443,
  "RMSE": 4.81203329609426,
  "pearson_r": 0.8845329720639536,
  "R2": 0.7165691421806206,
  "raw_MAE_uncorrected": 3.2667479177215193,
  "median_abs_error": 3.1070949808683537,
  "frac_abs_BAG_gt10": 0.04272151898734177,
  "BAG_vs_age_slope": 0.009338559385397212,
  "BAG_vs_age_p": 0.44525288473278424,
  "mean_BAG_CN": 0.01799950438948124,
  "age_group_acc_3class": 0.7816455696202531,
  "age_group_majority_baseline": 0.5780590717299579,
  "age_group_acc_CI95": [
    0.7567630016143911,
    0.8044342250172122
  ],
  "session_averaged_MAE": 3.759997448400704
}

By age band (CN):
   band   n  MAE  mean_BAG
  0-60 306 3.73     -0.90
 60-70 657 3.79     -0.25
 70-80 704 3.76      1.2

In [30]:
"""
05_noise_floor.py   (Phase 6 -- CPU)

Works on any predictions file that has: subject_id, mri_label, run, age, CDRTOT, BAG (+ optional predicted_age_sd, qc_flag,
scanner). Run it once per model variant (Step 6.8):
    python 05_noise_floor.py --variant m1
    python 05_noise_floor.py --bag path/to/brainage_predictions.csv --tag v2     # also works on your v2 file

Computes
  6.1 pooled within-session sigma_SEM (df-weighted; handles 3+ runs)         6.2 MDC95 = 1.96*sqrt(2)*sigma_SEM
  6.3 subject-cluster bootstrap CIs (1000)                                    6.4 ICC(A,1) from repeats + wide-cohort ICC
  6.5 heteroscedasticity (age, mean BAG, ensemble SD, QC)                     6.6 CN vs impaired: SEM ratio + CI, Levene
  6.7 uncertainty calibration (rank corr + quintile table)  and a robust sigma (MAD) + sigma without QC-flagged sessions,
      because heavy-tailed repeat differences (failed registrations) dominate a plain SD.
"""
import argparse, json
import numpy as np, pandas as pd
from scipy.stats import spearmanr, levene, f as fdist
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import config_v3 as C


def session_table(df):
    rows = []
    for lab, g in df.sort_values(["mri_label", "run"]).groupby("mri_label"):
        if len(g) < 2:
            continue
        b = g.BAG.to_numpy(float)
        rows.append(dict(mri_label=lab, subject_id=g.subject_id.iloc[0], age=g.age.mean(), CDRTOT=g.CDRTOT.iloc[0],
                         m=len(g), ss=float(((b - b.mean()) ** 2).sum()), df=len(g) - 1, mean_bag=b.mean(),
                         d12=b[0] - b[1], sd_within=float(b.std(ddof=1)),
                         ens_sd=float(g.predicted_age_sd.mean()) if "predicted_age_sd" in g else np.nan,
                         qc_flag=int(g.qc_flag.max()) if "qc_flag" in g else 0,
                         scanner=g.scanner.iloc[0] if "scanner" in g else None))
    return pd.DataFrame(rows)


def pooled_sem(s):
    return float(np.sqrt(s.ss.sum() / s.df.sum())) if len(s) and s.df.sum() > 0 else np.nan


def robust_sem(s):
    d = s.d12.to_numpy()
    return float(1.4826 * np.median(np.abs(d - np.median(d))) / np.sqrt(2)) if len(d) else np.nan


def boot(s, fn, n=1000, seed=C.SEED):
    rng = np.random.default_rng(seed)
    g = {k: v for k, v in s.groupby("subject_id")}
    keys = np.array(list(g))
    vals = [fn(pd.concat([g[k] for k in rng.choice(keys, len(keys), replace=True)])) for _ in range(n)]
    return [float(x) for x in np.nanpercentile(vals, [2.5, 97.5])]


def icc_from_pairs(df):
    rows = []
    for _, g in df.sort_values(["mri_label", "run"]).groupby("mri_label"):
        if len(g) >= 2:
            rows.append(g.BAG.to_numpy(float)[:2])
    X = np.array(rows)
    n, k = X.shape if len(rows) else (0, 2)
    if n < 5:
        return np.nan
    gm = X.mean()
    ssr = k * ((X.mean(1) - gm) ** 2).sum()
    ssc = n * ((X.mean(0) - gm) ** 2).sum()
    sst = ((X - gm) ** 2).sum()
    sse = sst - ssr - ssc
    msr, msc, mse = ssr / (n - 1), ssc / (k - 1), sse / ((n - 1) * (k - 1))
    return float((msr - mse) / (msr + (k - 1) * mse + k * (msc - mse) / n))


def wide_icc(df, sigma):
    first = df.sort_values("day").groupby("subject_id").first() if "day" in df else df.groupby("subject_id").first()
    vb = float(first.BAG.var(ddof=1)) - sigma ** 2
    return float(max(vb, 0) / (max(vb, 0) + sigma ** 2)), float(first.BAG.var(ddof=1))


def main(bag_path, tag, exclude_qc):
    df = pd.read_csv(bag_path)
    if exclude_qc and C.QC_CSV.exists() and "qc_flag" not in df:
        df = df.merge(pd.read_csv(C.QC_CSV)[["file_path", "qc_flag", "ncc_repeat"]], on="file_path", how="left")
    S = session_table(df)
    R = {"tag": tag, "n_repeat_sessions": int(len(S)), "n_subjects": int(S.subject_id.nunique()),
         "sessions_with_3plus_runs": int((S.m >= 3).sum())}

    # 6.1-6.3
    sig = pooled_sem(S)
    R.update(sigma_SEM=sig, sigma_SEM_CI=boot(S, pooled_sem), MDC95=1.96 * np.sqrt(2) * sig,
             MDC95_CI=[1.96 * np.sqrt(2) * x for x in boot(S, pooled_sem)])
    R["sigma_SEM_robust_MAD"] = robust_sem(S)
    R["n_sessions_abs_diff_gt5y"] = int((S.d12.abs() > 5).sum())
    if S.qc_flag.sum() > 0:
        R["sigma_SEM_excluding_QC_flagged"] = pooled_sem(S[S.qc_flag == 0])
        R["n_QC_flagged_repeat_sessions"] = int(S.qc_flag.sum())
    # 6.4 ICC
    R["ICC_A1_repeats"] = icc_from_pairs(df)
    R["ICC_wide_cohort"], R["between_subject_var_first_session"] = wide_icc(df, sig)
    # 6.5 heteroscedasticity
    het = {}
    for col in ["age", "mean_bag", "ens_sd"]:
        if S[col].notna().sum() > 5:
            rho, p = spearmanr(S[col], S.sd_within)
            het[col] = dict(rho=float(rho), p=float(p))
    R["heteroscedasticity_spearman_vs_within_SD"] = het
    bands = []
    for lo, hi in C.AGE_BANDS:
        g = S[(S.age >= lo) & (S.age < hi)]
        if len(g) >= 5:
            bands.append(dict(band=f"{lo}-{hi}", n_sessions=len(g), sigma_SEM=pooled_sem(g)))
    R["sigma_SEM_by_age_band"] = bands
    # 6.6 CN vs impaired
    cn, im = S[S.CDRTOT == 0], S[S.CDRTOT > 0]
    if len(cn) > 5 and len(im) > 5:
        R["sigma_SEM_CN"], R["sigma_SEM_impaired"] = pooled_sem(cn), pooled_sem(im)
        R["n_CN_sessions"], R["n_impaired_sessions"] = int(len(cn)), int(len(im))
        stat, p = levene(cn.d12, im.d12)
        R["levene_p"] = float(p)
        ratio = lambda s: pooled_sem(s[s.CDRTOT > 0]) / pooled_sem(s[s.CDRTOT == 0])
        R["SEM_ratio_impaired_over_CN"] = float(ratio(S))
        R["SEM_ratio_CI"] = boot(S, ratio)
        if S.scanner.notna().any():
            R["scanner_by_group"] = pd.crosstab(S.scanner, S.CDRTOT > 0).to_dict()
    # 6.7 calibration
    cal = {}
    for name, g in [("all", S), ("CN", cn), ("impaired", im)]:
        g = g.dropna(subset=["ens_sd"])
        if len(g) >= 10:
            rho, p = spearmanr(g.ens_sd, g.sd_within)
            cal[name] = dict(n=int(len(g)), spearman_rho=float(rho), p=float(p))
            q = pd.qcut(g.ens_sd, 5, duplicates="drop")
            tab = g.groupby(q, observed=True).agg(pred_spread=("ens_sd", "mean"), observed_sd=("sd_within", "mean"), n=("ens_sd", "size"))
            tab.to_csv(C.RES_DIR / f"calibration_{tag}_{name}.csv")
            if name == "all":
                plt.figure(figsize=(4, 4)); plt.plot(tab.pred_spread, tab.observed_sd, "o-")
                plt.xlabel("mean ensemble spread (y)"); plt.ylabel("mean observed within-session SD (y)")
                plt.title(f"Uncertainty calibration ({tag})"); plt.tight_layout()
                plt.savefig(C.RES_DIR / f"calibration_{tag}.png", dpi=200); plt.close()
    R["uncertainty_calibration"] = cal

    print(json.dumps(R, indent=2, default=lambda o: o if not isinstance(o, np.generic) else o.item()))
    C.RES_DIR.mkdir(parents=True, exist_ok=True)
    json.dump(R, open(C.RES_DIR / f"noise_floor_{tag}.json", "w"), indent=2, default=lambda o: o.item() if isinstance(o, np.generic) else str(o))
    S.to_csv(C.RES_DIR / f"repeat_sessions_{tag}.csv", index=False)


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--variant", default="m1")
    ap.add_argument("--bag", default=None)
    ap.add_argument("--tag", default=None)
    ap.add_argument("--exclude_qc", action="store_true", help="merge QC flags to report sigma without flagged sessions")
    a = ap.parse_args(args=["--exclude_qc"])
    C.RES_DIR.mkdir(parents=True, exist_ok=True)
    main(a.bag or str(C.PRED_DIR / f"bag_{a.variant}.csv"), a.tag or a.variant, a.exclude_qc)


{
  "tag": "m1",
  "n_repeat_sessions": 777,
  "n_subjects": 606,
  "sessions_with_3plus_runs": 18,
  "sigma_SEM": 0.6697957172632136,
  "sigma_SEM_CI": [
    0.6238242891350868,
    0.7190488289878315
  ],
  "MDC95": 1.8565790072511814,
  "MDC95_CI": [
    1.7291527096558856,
    1.9931016676876439
  ],
  "sigma_SEM_robust_MAD": 0.5140644594945388,
  "n_sessions_abs_diff_gt5y": 0,
  "ICC_A1_repeats": 0.9850012793427525,
  "ICC_wide_cohort": 0.9855816506939192,
  "between_subject_var_first_session": 31.11495590378981,
  "heteroscedasticity_spearman_vs_within_SD": {
    "age": {
      "rho": -0.21615559826510855,
      "p": 1.1455332672195922e-09
    },
    "mean_bag": {
      "rho": 0.02811665321272297,
      "p": 0.4338404398707272
    },
    "ens_sd": {
      "rho": 0.12550477869959492,
      "p": 0.00045378137466063926
    }
  },
  "sigma_SEM_by_age_band": [
    {
      "band": "0-60",
      "n_sessions": 120,
      "sigma_SEM": 0.8809305596820723
    },
    {
      "band": "60-70",

In [31]:
"""
06_slopes.py   (Phases 7-8 -- CPU)

Builds baseline windows, raw / thresholded / shrunk / SNR slopes, slope SE, reliability, and writes the survival cohort.

    python 06_slopes.py --variant m1 --k 3 --min_span 2.0
    python 06_slopes.py --bag v3/predictions/bag_m1.csv --noise_json v3/results/noise_floor_m1.json --k 3

sigma (residual SD of a single scan's BAG) comes from 05_noise_floor.py (pooled sigma_SEM) unless --sigma is given.
IMPORTANT caveat built into the output: same-day sigma_SEM is a LOWER BOUND on between-visit noise (no repositioning
across months, no scanner drift, no true non-linearity). The unconstrained mixed model below estimates the residual SD
actually present in the windows; if it is much larger than sigma_SEM, say so and run the sensitivity analysis in 08.
"""
import argparse, json
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import warnings
warnings.filterwarnings("ignore")
import config_v3 as C
import longitudinal_lib as L


def main(a):
    bag = pd.read_csv(a.bag or C.PRED_DIR / f"bag_{a.variant}.csv")
    if a.sigma is None:
        nj = json.load(open(a.noise_json or C.RES_DIR / f"noise_floor_{a.variant}.json"))
        sigma = nj["sigma_SEM"]
    else:
        sigma = a.sigma
    print(f"sigma_SEM used = {sigma:.3f} y")
    sessions = L.build_sessions(bag)
    d, post = L.build_windows(sessions, k=a.k, min_span_years=a.min_span, sigma=sigma)
    if len(d) < 10:
        raise SystemExit(f"Only {len(d)} eligible subjects for k={a.k}, min_span={a.min_span}")
    d, info = L.add_slope_variants(d, sigma)
    info.update(k=a.k, min_span=a.min_span, sigma=sigma, n_events=int(d.event.sum()),
                median_window_span=float(d.window_span_years.median()),
                n_subjects_before_span_rule=int(L.build_windows(sessions, k=a.k, min_span_years=0, sigma=sigma)[0].shape[0]))

    # slope reliability headline
    info["reliability_flag"] = "LOW (<0.5): slope is close to unusable as an individual marker" if info["mean_reliability"] < 0.5 else "adequate (>=0.5)"
    info["SE_larger_than_abs_slope"] = bool(info["median_SE"] > info["median_abs_slope"])

    # unconstrained mixed model on the SAME baseline-window sessions (sensitivity; Step 8.4 caveat)
    try:
        ws = []
        for sid in d.subject_id:
            g = sessions[sessions.subject_id == sid].sort_values("day").iloc[:a.k].copy()
            g["t"] = (g.day - g.day.iloc[0]) / 365.25
            ws.append(g)
        ws = pd.concat(ws)
        mm = smf.mixedlm("BAG ~ t", ws, groups=ws["subject_id"], re_formula="~t").fit(reml=True)
        info["mixedlm_residual_SD_free"] = float(np.sqrt(mm.scale))
        info["mixedlm_slope_SD_free"] = float(np.sqrt(mm.cov_re.iloc[1, 1]))
        info["residual_SD_free_over_sigma_SEM"] = float(np.sqrt(mm.scale) / sigma)
    except Exception as e:
        info["mixedlm_error"] = str(e)[:200]

    tag = f"{a.variant}_k{a.k}" + (f"_span{a.min_span:g}" if a.min_span else "")
    C.RES_DIR.mkdir(parents=True, exist_ok=True)
    d.to_csv(C.RES_DIR / f"slopes_{tag}.csv", index=False)
    pd.to_pickle(post, C.RES_DIR / f"post_{tag}.pkl")
    json.dump(info, open(C.RES_DIR / f"slope_info_{tag}.json", "w"), indent=2)
    print(json.dumps(info, indent=2))

    fig, ax = plt.subplots(1, 2, figsize=(8, 3.6))
    ax[0].scatter(d.S_raw, d.S_shrunk, s=12); lim = [d.S_raw.min(), d.S_raw.max()]
    ax[0].plot(lim, lim, "k--", lw=.8); ax[0].set_xlabel("raw slope (y/y)"); ax[0].set_ylabel("shrunk slope (y/y)")
    ax[1].scatter(d.S_raw.abs(), d.SE_S, s=12); ax[1].plot([0, d.S_raw.abs().max()], [0, d.S_raw.abs().max()], "k--", lw=.8)
    ax[1].set_xlabel("|raw slope|"); ax[1].set_ylabel("slope SE")
    plt.tight_layout(); plt.savefig(C.RES_DIR / f"slopes_{tag}.png", dpi=200)
    print("saved", C.RES_DIR / f"slopes_{tag}.csv")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--variant", default="m1")
    ap.add_argument("--bag", default=None)
    ap.add_argument("--noise_json", default=None)
    ap.add_argument("--sigma", type=float, default=None)
    ap.add_argument("--k", type=int, default=3)
    ap.add_argument("--min_span", type=float, default=0.0)
    main(ap.parse_args(args=["--variant", "m1", "--k", "3", "--min_span", "2.0"]))


sigma_SEM used = 0.670 y
{
  "tau2": 0.21906133349303455,
  "tau": 0.4680398845109619,
  "mu": 0.12864040529496115,
  "mean_reliability": 0.883420745559337,
  "median_SE": 0.1549276946339639,
  "median_abs_slope": 0.3746613767514711,
  "frac_exceeding_threshold": 0.4857142857142857,
  "n": 35,
  "corr_raw_shrunk": 0.9985990928824158,
  "k": 3,
  "min_span": 2.0,
  "sigma": 0.6697957172632136,
  "n_events": 2,
  "median_window_span": 5.292265571526352,
  "n_subjects_before_span_rule": 36,
  "reliability_flag": "adequate (>=0.5)",
  "SE_larger_than_abs_slope": false,
  "mixedlm_residual_SD_free": 1.1428912478137547,
  "mixedlm_slope_SD_free": 0.5804474010962054,
  "residual_SD_free_over_sigma_SEM": 1.7063280913225456
}
saved v3/results/slopes_m1_k3_span2.csv


In [32]:
"""
07_survival_validation.py   (Phase 9 -- CPU)

    python 07_survival_validation.py --slopes v3/results/slopes_m1_k3_span2.csv [--clinical extra.csv] [--n_repeats 20] [--n_boot 1000]

Primary-endpoint rule (pre-registered): >= 30 events -> survival is primary, continuous endpoint parallel;
fewer events -> continuous endpoint primary, survival exploratory. Printed at the top.

Models (all share age_base, sex, bag_base = "floor"):
    floor | floor+raw | floor+shrunk | floor+raw+shrunk   (+ optional clinical block: baseline MMSE, APOE4, hippocampus, ventricles ...)
Reported:  likelihood-ratio tests (nested), repeated stratified subject-level CV C-index (ridge-Cox, penalty by inner CV),
           paired OUT-OF-BAG bootstrap of  C(shrunk) - C(raw),  time-dependent AUC at 2 and 4 y, Schoenfeld PH test,
           continuous endpoint (future CDR-SB / MMSE slope) with HC3 standard errors.
"""
import argparse, json, warnings
import numpy as np, pandas as pd
from scipy.stats import chi2
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold
from lifelines import CoxPHFitter
from lifelines.utils import concordance_index
from lifelines.statistics import proportional_hazard_test
import statsmodels.api as sm
import config_v3 as C
import longitudinal_lib as L

warnings.filterwarnings("ignore")
T, E = "time_to_event", "event"
FLOOR = ["age_base", "sex", "bag_base"]
GRID = [0.01, 0.1, 0.5, 1.0]


def fit_cox(train, cols, pen):
    mu, sd = train[cols].mean(), train[cols].std(ddof=0).replace(0, 1)
    z = ((train[cols] - mu) / sd).assign(**{T: train[T].values, E: train[E].values})
    m = CoxPHFitter(penalizer=pen, l1_ratio=0.0).fit(z, T, E)
    return m, mu, sd


def risk(model, mu, sd, df, cols):
    return model.predict_partial_hazard(((df[cols] - mu) / sd)).to_numpy()


def cidx(model, mu, sd, df, cols):
    return concordance_index(df[T], -risk(model, mu, sd, df, cols), df[E])


def tune_pen(train, cols, seed=0):
    skf = StratifiedKFold(3, shuffle=True, random_state=seed)
    best, bp = -1, GRID[-1]
    for pen in GRID:
        cs = []
        for tr, te in skf.split(train, train[E]):
            try:
                m, mu, sd = fit_cox(train.iloc[tr], cols, pen)
                if train.iloc[te][E].sum() >= 2:
                    cs.append(cidx(m, mu, sd, train.iloc[te], cols))
            except Exception:
                pass
        if cs and np.mean(cs) > best:
            best, bp = np.mean(cs), pen
    return bp


def td_auc(t_arr, e_arr, rk, horizon):
    case = (t_arr <= horizon) & (e_arr == 1)
    ctrl = t_arr > horizon
    if case.sum() < 3 or ctrl.sum() < 3:
        return np.nan
    a, b = rk[case][:, None], rk[ctrl][None, :]
    return float(((a > b).sum() + 0.5 * (a == b).sum()) / (a.size * b.shape[1]))


def build_models(df, clinical_cols):
    M = {"floor": FLOOR, "floor+raw": FLOOR + ["S_raw"], "floor+shrunk": FLOOR + ["S_shrunk"],
         "floor+raw+shrunk": FLOOR + ["S_raw", "S_shrunk"]}
    if clinical_cols:
        M["clinical"] = FLOOR + clinical_cols
        M["clinical+shrunk"] = FLOOR + clinical_cols + ["S_shrunk"]
    return M


def survival_block(df, models, n_repeats, n_boot, out):
    ev = int(df[E].sum())
    out["n_subjects"], out["n_events"] = len(df), ev
    # --- LRT (unpenalized, full data, nested)
    ll = {}
    for name, cols in models.items():
        try:
            ll[name] = (CoxPHFitter().fit(df[cols + [T, E]], T, E).log_likelihood_, len(cols))
        except Exception as e:
            ll[name] = (np.nan, len(cols))
    lrt = {}
    for small, big in [("floor", "floor+raw"), ("floor", "floor+shrunk"), ("floor+raw", "floor+raw+shrunk"),
                       ("floor+shrunk", "floor+raw+shrunk"), ("clinical", "clinical+shrunk")]:
        if small in ll and big in ll and np.isfinite(ll[small][0]) and np.isfinite(ll[big][0]):
            stat = 2 * (ll[big][0] - ll[small][0]); k = ll[big][1] - ll[small][1]
            lrt[f"{small} -> {big}"] = dict(chi2=float(stat), df=int(k), p=float(chi2.sf(max(stat, 0), k)))
    out["LRT_unpenalized"] = lrt
    out["unpenalized_covariates_per_10_events_ok"] = {n: bool(len(c) <= ev / 10) for n, c in models.items()}

    # --- repeated stratified subject-level CV
    rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=n_repeats, random_state=C.SEED)
    rows, oof_risk = [], {n: np.full(len(df), np.nan) for n in models}
    for fi, (tr, te) in enumerate(rskf.split(df, df[E])):
        dtr, dte = df.iloc[tr], df.iloc[te]
        if dte[E].sum() < 2:
            continue
        row = {"fold": fi}
        for name, cols in models.items():
            try:
                pen = tune_pen(dtr, cols, seed=fi)
                m, mu, sd = fit_cox(dtr, cols, pen)
                row[name] = cidx(m, mu, sd, dte, cols)
                if fi < 5:                                   # first repeat -> pooled OOF risk for AUC(t)
                    oof_risk[name][te] = risk(m, mu, sd, dte, cols)
            except Exception:
                row[name] = np.nan
        rows.append(row)
    cv = pd.DataFrame(rows)
    cv.to_csv(C.RES_DIR / "cv_cindex.csv", index=False)
    out["CV_cindex_mean"] = {n: float(cv[n].mean()) for n in models}
    out["CV_cindex_sd_across_folds"] = {n: float(cv[n].std()) for n in models}
    if {"floor+raw", "floor+shrunk"} <= set(cv.columns):
        dlt = (cv["floor+shrunk"] - cv["floor+raw"]).dropna()
        out["CV_paired_diff_shrunk_minus_raw"] = dict(mean=float(dlt.mean()), pct2_5=float(np.percentile(dlt, 2.5)),
                                                      pct97_5=float(np.percentile(dlt, 97.5)),
                                                      note="spread over CV folds; folds overlap, so use the bootstrap below for inference")
    # --- time-dependent AUC on pooled first-repeat OOF risks
    auc = {}
    for n in models:
        r = oof_risk[n]; ok = ~np.isnan(r)
        auc[n] = {f"AUC_{h}y": td_auc(df[T].values[ok], df[E].values[ok], r[ok], h) for h in (2, 4)}
    out["time_dependent_AUC_naive_no_IPCW"] = auc

    # --- paired OUT-OF-BAG bootstrap: shrunk vs raw (identical subjects in each resample)
    rng = np.random.default_rng(C.SEED)
    pens = {n: tune_pen(df, models[n]) for n in ("floor+raw", "floor+shrunk")}
    diffs = []
    for b in range(n_boot):
        idx = rng.integers(0, len(df), len(df))
        oob = np.setdiff1d(np.arange(len(df)), idx)
        dtr, dte = df.iloc[idx], df.iloc[oob]
        if dtr[E].sum() < 5 or dte[E].sum() < 3:
            continue
        try:
            cr = cidx(*fit_cox(dtr, models["floor+raw"], pens["floor+raw"]), dte, models["floor+raw"])
            cs = cidx(*fit_cox(dtr, models["floor+shrunk"], pens["floor+shrunk"]), dte, models["floor+shrunk"])
            diffs.append(cs - cr)
        except Exception:
            pass
    diffs = np.array(diffs)
    if len(diffs) >= 100:
        out["PRIMARY_paired_OOB_bootstrap_C_shrunk_minus_raw"] = dict(
            n_valid=int(len(diffs)), mean=float(diffs.mean()), CI95=[float(x) for x in np.percentile(diffs, [2.5, 97.5])],
            excludes_zero=bool(np.percentile(diffs, 2.5) > 0 or np.percentile(diffs, 97.5) < 0))
    else:
        out["PRIMARY_paired_OOB_bootstrap_C_shrunk_minus_raw"] = f"only {len(diffs)} valid resamples -- too few events"
    # --- PH assumption
    try:
        m = CoxPHFitter().fit(df[models["floor+shrunk"] + [T, E]], T, E)
        ph = proportional_hazard_test(m, df[models["floor+shrunk"] + [T, E]], time_transform="rank").summary
        out["schoenfeld_p"] = {k: float(v) for k, v in ph["p"].items()}
        out["hazard_ratio_shrunk_per_unit_slope"] = float(np.exp(m.params_["S_shrunk"]))
    except Exception as e:
        out["schoenfeld_p"] = f"failed: {str(e)[:80]}"


def continuous_block(df, post, out):
    res = {}
    for endpoint, base_col in [("CDRSUM", "base_CDRSUM"), ("MMSE", "base_MMSE")]:
        fut = L.future_clinical_slopes(post, endpoint)
        d = df.set_index("subject_id").join(fut, how="inner").dropna(subset=[fut.name])
        d["baseline_score"] = [post[s][base_col] for s in d.index]
        covs = FLOOR + ([ "baseline_score"] if d.baseline_score.std() > 0 else [])
        d = d.dropna(subset=covs + ["S_raw", "S_shrunk"])
        if len(d) < 15:
            res[endpoint] = f"only {len(d)} subjects with >=2 post-window sessions"
            continue
        y = d[fut.name]
        base = sm.OLS(y, sm.add_constant(d[covs])).fit()
        r = dict(n=int(len(d)), R2_covariates_only=float(base.rsquared))
        for name in ("S_raw", "S_shrunk"):
            X = sm.add_constant(d[covs + [name]])
            m = sm.OLS(y, X).fit(cov_type="HC3")
            ry = sm.OLS(y, sm.add_constant(d[covs])).fit().resid
            rx = sm.OLS(d[name], sm.add_constant(d[covs])).fit().resid
            r[name] = dict(coef=float(m.params[name]), se_HC3=float(m.bse[name]), p=float(m.pvalues[name]),
                           R2=float(m.rsquared), delta_R2=float(m.rsquared - base.rsquared),
                           partial_r=float(np.corrcoef(rx, ry)[0, 1]))
        res[endpoint] = r
    out["continuous_endpoint"] = res


def main(a):
    d = pd.read_csv(a.slopes)
    tag = a.slopes.split("slopes_")[-1].replace(".csv", "")
    post = pd.read_pickle(a.slopes.replace("slopes_", "post_").replace(".csv", ".pkl"))
    clin = []
    if a.clinical:
        extra = pd.read_csv(a.clinical)                        # subject_id + numeric baseline columns (apoe4, hippo, vent, ...)
        d = d.merge(extra, on="subject_id", how="left")
        clin = [c for c in extra.columns if c != "subject_id"]
        d = d.dropna(subset=clin)
    d["baseline_MMSE_win"] = [post[s]["base_MMSE"] for s in d.subject_id]
    if d.baseline_MMSE_win.notna().all() and "baseline_MMSE" not in clin and a.use_mmse:
        clin = clin + ["baseline_MMSE_win"]
    ev = int(d[E].sum())
    rule = "SURVIVAL primary (>=30 events), continuous parallel" if ev >= 30 else "CONTINUOUS primary (<30 events), survival EXPLORATORY"
    print(f"events={ev} of {len(d)} subjects -> {rule}")
    out = dict(tag=tag, primary_endpoint_rule=rule)
    cv_se = float(d.SE_S.std() / d.SE_S.mean())
    out["SE_coefficient_of_variation"] = cv_se
    out["shrinkage_invariance_note"] = (
        "Shrunk = mu + r_i*(raw - mu). If SE_S is similar across subjects (CV of SE small), r_i is ~constant, shrunk is an affine "
        "copy of raw, and a Cox model/C-index cannot distinguish them (expected paired difference ~0). "
        "A shrunk-vs-raw C-index gain is only possible through heterogeneity in SE (different spans / visit counts). "
        f"Observed CV(SE) = {cv_se:.2f}.")
    print(out["shrinkage_invariance_note"])
    if ev >= 8:
        survival_block(d, build_models(d, clin), a.n_repeats, a.n_boot, out)
    else:
        out["survival"] = "fewer than 8 events: Cox models not fitted"
    continuous_block(d, post, out)
    C.RES_DIR.mkdir(parents=True, exist_ok=True)
    json.dump(out, open(C.RES_DIR / f"validation_{tag}.json", "w"), indent=2, default=lambda o: o.item() if isinstance(o, np.generic) else str(o))
    print(json.dumps(out, indent=2, default=lambda o: o.item() if isinstance(o, np.generic) else str(o)))


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--slopes", required=True)
    ap.add_argument("--clinical", default=None)
    ap.add_argument("--use_mmse", action="store_true", help="add baseline MMSE to the clinical block")
    ap.add_argument("--n_repeats", type=int, default=20)
    ap.add_argument("--n_boot", type=int, default=1000)
    main(ap.parse_args(args=["--slopes", "v3/results/slopes_m1_k3_span2.csv"]))


events=2 of 35 subjects -> CONTINUOUS primary (<30 events), survival EXPLORATORY
Shrunk = mu + r_i*(raw - mu). If SE_S is similar across subjects (CV of SE small), r_i is ~constant, shrunk is an affine copy of raw, and a Cox model/C-index cannot distinguish them (expected paired difference ~0). A shrunk-vs-raw C-index gain is only possible through heterogeneity in SE (different spans / visit counts). Observed CV(SE) = 0.26.
{
  "tag": "m1_k3_span2",
  "primary_endpoint_rule": "CONTINUOUS primary (<30 events), survival EXPLORATORY",
  "SE_coefficient_of_variation": 0.2569619493624806,
  "shrinkage_invariance_note": "Shrunk = mu + r_i*(raw - mu). If SE_S is similar across subjects (CV of SE small), r_i is ~constant, shrunk is an affine copy of raw, and a Cox model/C-index cannot distinguish them (expected paired difference ~0). A shrunk-vs-raw C-index gain is only possible through heterogeneity in SE (different spans / visit counts). Observed CV(SE) = 0.26.",
  "survival": "fewer than 8 

In [33]:
"""
08_robustness.py   (Phase 10 -- CPU)

Re-runs the slope + survival pipeline over a grid and writes one robustness table (v3/results/robustness_<variant>.csv):
    10.1 window size k in {3,4,5}          10.2 sigma at its bootstrap CI bounds     10.3 minimum window span {0,1,2} y
    10.4 sigma estimated on subjects OUTSIDE the survival cohort (and random split-halves) to show the noise floor is not
         driven by the subjects being analysed
    10.6 estimators: raw, thresholded, shrunk, SNR (C-index by repeated CV, same folds)
Not automated here (needs metadata you may not have): 10.5 scanner/field-strength strata, 10.7 alternative CDR-matching rule.

    python 08_robustness.py --variant m1
"""
import argparse, importlib, json, warnings
import numpy as np, pandas as pd
from sklearn.model_selection import RepeatedStratifiedKFold
warnings.filterwarnings("ignore")
import config_v3 as C
import longitudinal_lib as L
V = importlib.import_module("07_survival_validation")
N = importlib.import_module("05_noise_floor")


def quick(d, n_repeats):
    ev = int(d.event.sum())
    r = dict(n=len(d), events=ev)
    if ev < 8:
        return r
    models = {"floor": V.FLOOR, "raw": V.FLOOR + ["S_raw"], "thr": V.FLOOR + ["S_thr"], "shrunk": V.FLOOR + ["S_shrunk"], "snr": V.FLOOR + ["S_snr"]}
    rs = RepeatedStratifiedKFold(n_splits=5, n_repeats=n_repeats, random_state=C.SEED)
    acc = {m: [] for m in models}
    for fi, (tr, te) in enumerate(rs.split(d, d.event)):
        if d.iloc[te].event.sum() < 2:
            continue
        for m, cols in models.items():
            try:
                pen = V.tune_pen(d.iloc[tr], cols, seed=fi)
                acc[m].append(V.cidx(*V.fit_cox(d.iloc[tr], cols, pen), d.iloc[te], cols))
            except Exception:
                pass
    for m in models:
        r[f"C_{m}"] = float(np.mean(acc[m])) if acc[m] else np.nan
    return r


def main(a):
    bag = pd.read_csv(a.bag or C.PRED_DIR / f"bag_{a.variant}.csv")
    nj = json.load(open(a.noise_json or C.RES_DIR / f"noise_floor_{a.variant}.json"))
    sess = L.build_sessions(bag)
    sigmas = {"point": nj["sigma_SEM"], "ci_low": nj["sigma_SEM_CI"][0], "ci_high": nj["sigma_SEM_CI"][1]}
    rows = []
    for k in (3, 4, 5):
        for span in (0.0, 1.0, 2.0):
            for sname, sg in sigmas.items():
                if sname != "point" and (k != 3 or span != a.primary_span):
                    continue                                    # CI-bound sweep only around the primary configuration
                d, _ = L.build_windows(sess, k=k, min_span_years=span, sigma=sg)
                if len(d) < 15:
                    rows.append(dict(k=k, min_span=span, sigma=sname, n=len(d), note="too few subjects"))
                    continue
                d, info = L.add_slope_variants(d, sg)
                r = dict(k=k, min_span=span, sigma=sname, sigma_value=sg, mean_reliability=info["mean_reliability"],
                         frac_exceeding_threshold=info["frac_exceeding_threshold"]); r.update(quick(d, a.n_repeats))
                rows.append(r); print(r)
    # 10.4 sigma from subjects outside the survival cohort + split-half stability
    d3, _ = L.build_windows(sess, k=3, min_span_years=a.primary_span, sigma=nj["sigma_SEM"])
    S = N.session_table(bag)
    out_cohort = S[~S.subject_id.isin(d3.subject_id)]
    extra = dict(sigma_outside_survival_cohort=N.pooled_sem(out_cohort) if len(out_cohort) > 20 else np.nan,
                 n_sessions_outside=int(len(out_cohort)))
    rng = np.random.default_rng(C.SEED); subs = S.subject_id.unique()
    halves = []
    for _ in range(200):
        h = set(rng.choice(subs, len(subs) // 2, replace=False))
        halves.append(N.pooled_sem(S[S.subject_id.isin(h)]))
    extra.update(sigma_split_half_median=float(np.median(halves)), sigma_split_half_IQR=[float(x) for x in np.percentile(halves, [25, 75])])
    print(extra)
    pd.DataFrame(rows).to_csv(C.RES_DIR / f"robustness_{a.variant}.csv", index=False)
    json.dump(extra, open(C.RES_DIR / f"robustness_sigma_{a.variant}.json", "w"), indent=2)


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--variant", default="m1")
    ap.add_argument("--bag", default=None)
    ap.add_argument("--noise_json", default=None)
    ap.add_argument("--primary_span", type=float, default=2.0)
    ap.add_argument("--n_repeats", type=int, default=5)
    main(ap.parse_args(args=[]))


{'k': 3, 'min_span': 0.0, 'sigma': 'point', 'sigma_value': 0.6697957172632136, 'mean_reliability': 0.8828249682944055, 'frac_exceeding_threshold': 0.5, 'n': 36, 'events': 2}
{'k': 3, 'min_span': 1.0, 'sigma': 'point', 'sigma_value': 0.6697957172632136, 'mean_reliability': 0.8828249682944055, 'frac_exceeding_threshold': 0.5, 'n': 36, 'events': 2}
{'k': 3, 'min_span': 2.0, 'sigma': 'point', 'sigma_value': 0.6697957172632136, 'mean_reliability': 0.883420745559337, 'frac_exceeding_threshold': 0.4857142857142857, 'n': 35, 'events': 2}
{'k': 3, 'min_span': 2.0, 'sigma': 'ci_low', 'sigma_value': 0.6238242891350868, 'mean_reliability': 0.898249705526169, 'frac_exceeding_threshold': 0.5428571428571428, 'n': 35, 'events': 2}
{'k': 3, 'min_span': 2.0, 'sigma': 'ci_high', 'sigma_value': 0.7190488289878315, 'mean_reliability': 0.8665568509676157, 'frac_exceeding_threshold': 0.45714285714285713, 'n': 35, 'events': 2}
{'sigma_outside_survival_cohort': 0.681848505405787, 'n_sessions_outside': 706, 'si

In [34]:
"""
09_leakage_audit.py   (Phase 11 -- run BEFORE any number goes into the paper)

Automated where possible, otherwise prints what to confirm by hand.
    python 09_leakage_audit.py --variant m1 --slopes v3/results/slopes_m1_k3_span2.csv
"""
import argparse, glob, json
import numpy as np, pandas as pd
import config_v3 as C
import longitudinal_lib as L

res = []
def check(name, ok, detail=""):
    res.append((name, "PASS" if ok else "FAIL", detail)); print(f"[{'PASS' if ok else 'FAIL'}] {name} {detail}")


def main(a):
    man = pd.read_csv(C.MANIFEST_CSV)
    # 1 no subject in two folds
    check("1. every subject belongs to exactly one fold (repeats + later visits included)",
          man.groupby("subject_id").fold.nunique().max() == 1)
    # 7 early stopping never touched the held-out fold; 6 consistency pairs only from training subjects
    ok, n = True, 0
    for f in glob.glob(str(C.CKPT_DIR / "*.json")):
        meta = json.load(open(f)); tag = meta["tag"]; k = int(tag.split("_f")[1].split("_")[0])
        held = set(man[man.fold == k].subject_id)
        n += 1
        if set(meta["val_subjects"]) & held:
            ok = False; print("   leak in", tag)
    check("7. early-stopping subjects never belong to the held-out fold", ok and n > 0, f"({n} checkpoints inspected)")
    check("6. consistency-loss pairs come from training subjects only", True, "(enforced by assert in 03_train_cv.py: tr.fold != held-out fold)")
    # 2 bias-correction parameters from other folds only
    j = C.RES_DIR / f"accuracy_{a.variant}.json"
    if j.exists():
        bp = json.load(open(j))["bias_params_per_fold"]
        n_cn_sessions = man[man.CDRTOT == 0].mri_label.nunique()
        check("2. bias correction fitted on other folds only (cross-fitted)", all(v["n_sessions"] < n_cn_sessions for v in bp.values()), f"({len(bp)} folds)")
    else:
        check("2. bias correction cross-fitted", False, "accuracy json missing -- run 04b")
    # 3/4 survival features contain no post-window information: scramble post-window BAG, slopes must not move
    bag = pd.read_csv(C.PRED_DIR / f"bag_{a.variant}.csv")
    sess = L.build_sessions(bag)
    sig = json.load(open(C.RES_DIR / f"noise_floor_{a.variant}.json"))["sigma_SEM"]
    d0, _ = L.build_windows(sess, k=a.k, min_span_years=a.min_span, sigma=sig)
    s2 = sess.copy(); rng = np.random.default_rng(0)
    s2["rank"] = s2.groupby("subject_id").day.rank(method="first")
    m = s2["rank"] > a.k
    s2.loc[m, "BAG"] = rng.normal(0, 50, m.sum())               # destroy every post-window BAG value
    d1, _ = L.build_windows(s2, k=a.k, min_span_years=a.min_span, sigma=sig)
    mrg = d0.merge(d1, on="subject_id", suffixes=("_a", "_b"))
    same = all(np.allclose(mrg[c + "_a"], mrg[c + "_b"]) for c in ("S_raw", "bag_base", "SE_S", "age_base"))
    check("3/4. slopes, baseline BAG and SE use baseline-window sessions only (post-window BAG scrambled -> unchanged)", same and len(mrg) == len(d0))
    # 5, 8 manual / logged
    env = C.OUT / "environment_v3.json"
    check("8. seeds and software versions logged", env.exists(), f"(seed={C.SEED}; checkpoints tagged variant_fold_seed)")
    print("\nMANUAL: 5. confirm VARIANTS in config_v3.py and PREREGISTRATION_v3.md are unchanged since before survival results.")
    print("MANUAL: confirm QC exclusion rules were fixed from image properties only (never from prediction error).")
    print("MANUAL: confirm the v2 TEST split was not used to choose any v3 hyper-parameter (v3 decisions: folds + probe on fold 0).")
    pd.DataFrame(res, columns=["check", "status", "detail"]).to_csv(C.RES_DIR / "leakage_audit.csv", index=False)
    if any(r[1] == "FAIL" for r in res):
        raise SystemExit("AUDIT FAILED")

if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--variant", default="m1")
    ap.add_argument("--k", type=int, default=3)
    ap.add_argument("--min_span", type=float, default=0.0)
    main(ap.parse_args(args=[]))


[PASS] 1. every subject belongs to exactly one fold (repeats + later visits included) 
[PASS] 7. early-stopping subjects never belong to the held-out fold (25 checkpoints inspected)
[PASS] 6. consistency-loss pairs come from training subjects only (enforced by assert in 03_train_cv.py: tr.fold != held-out fold)
[PASS] 2. bias correction fitted on other folds only (cross-fitted) (5 folds)
[PASS] 3/4. slopes, baseline BAG and SE use baseline-window sessions only (post-window BAG scrambled -> unchanged) 
[PASS] 8. seeds and software versions logged (seed=42; checkpoints tagged variant_fold_seed)

MANUAL: 5. confirm VARIANTS in config_v3.py and PREREGISTRATION_v3.md are unchanged since before survival results.
MANUAL: confirm QC exclusion rules were fixed from image properties only (never from prediction error).
MANUAL: confirm the v2 TEST split was not used to choose any v3 hyper-parameter (v3 decisions: folds + probe on fold 0).
